# domain common support

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Domain overlap and common-support evaluation

This notebook tests whether the FLAMINGO-to-TNG transfer errors found are concentrated in regions where the simulations are easy to distinguish.
It trains a domain classifier for each of the five unique dynamics-block feature
spaces and assigns every pair an out-of-fold score. These scores are then joined to
the per-pair posterior summaries produced by Roihu job 1521132.

The calculation is evaluation-only. It does not fit or update an NPE. It also does not
define one scientifically authoritative overlap threshold. Instead, the posterior
metrics and retained sample counts are reported over a sequence of score intervals
around the balanced domain boundary.

The central question is conditional. If transfer bias becomes small inside regions
where simulation membership is difficult to predict, lack of feature-space support is
an important part of the failure. If the bias remains inside common support, marginal
overlap alone is insufficient and a simulation-dependent conditional mapping remains.
Neither outcome, by itself, identifies a causal astrophysical mechanism.


## 1. Configuration

The default run uses five outer folds. In every outer fold, connected host components
are held out together. A separate group-aware validation split calibrates the domain
score under equal simulation weights; the outer test fold is not used for fitting or
calibration. Consequently, every score used below comes from a classifier that did
not see that pair or another pair in its connected host component.

`0.4 <= score <= 0.6` is included as one sensitivity interval, not as a privileged
definition. Wider and narrower intervals must be inspected together with their sample
counts, especially for the 277 different-host TNG systems.


In [ ]:
RANDOM_SEED = 260924
OUTER_FOLDS = 5
VALIDATION_FOLDS = 5
CLASSIFIER_MAX_ITER = 160
OVERLAP_HALF_WIDTHS = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import subprocess
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

# Avoid joblib's Windows physical-core probe when only logical-core information is
# available. This does not change the statistical calculation.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, r2_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold


def find_project_root() -> Path:
    candidates = []
    if os.environ.get("LG_PROJECT"):
        candidates.append(Path(os.environ["LG_PROJECT"]))
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "src" / "lg_npe").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the FLAMINGO project. Set LG_PROJECT to the project path."
    )


PROJECT = find_project_root()
sys.path.insert(0, str(PROJECT / "src"))

POSTERIOR_EVALUATION = (
    PROJECT
    / "out"
    / "mass_comparison"
    / "posterior_evaluation"
    / "20260924T081006Z"
)
if not POSTERIOR_EVALUATION.is_dir():
    raise FileNotFoundError(
        "The job-1521132 posterior export is missing: "
        f"{POSTERIOR_EVALUATION}"
    )

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = PROJECT / "out" / "common_support" / "domain_common_support" / timestamp
OUTPUT.mkdir(parents=True, exist_ok=False)

FEATURE_SPACES = OrderedDict([
    ("dynamics", "none"),
    ("bound_subhalo_ratio", "bound_subhalo_ratio"),
    ("stellar_individual", "stellar_individual"),
    ("stellar_ratio", "stellar_ratio"),
    ("stellar_sum", "stellar_sum"),
])

MODEL_TO_FEATURE_SPACE = {
    "m200c_bound_ratio": "bound_subhalo_ratio",
    "m200c_dynamics": "dynamics",
    "m200c_stellar_individual": "stellar_individual",
    "m200c_stellar_ratio": "stellar_ratio",
    "m200c_stellar_sum": "stellar_sum",
    "bound_sum_dynamics": "dynamics",
    "bound_sum_stellar_individual": "stellar_individual",
    "bound_sum_stellar_ratio": "stellar_ratio",
    "bound_sum_stellar_sum": "stellar_sum",
}
MODEL_TARGET = {
    label: ("m200c" if label.startswith("m200c_") else "bound_subhalo_sum")
    for label in MODEL_TO_FEATURE_SPACE
}

try:
    git_revision = subprocess.run(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    git_revision = None

posterior_manifest_path = POSTERIOR_EVALUATION / "evaluation_manifest.json"
posterior_manifest_sha256 = hashlib.sha256(
    posterior_manifest_path.read_bytes()
).hexdigest()

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": metadata.version("matplotlib"),
}
print(f"Project: {PROJECT}")
print(f"Posterior export: {POSTERIOR_EVALUATION}")
print(f"Output: {OUTPUT}")
print(json.dumps(versions, indent=2))


## 2. Authoritative populations and feature spaces

The same global complete-case policy used by the NPE follow-up is included. The domain
label is never included as an NPE feature, and the halo-mass target is used only to
reproduce the approved population mask. The five classifier inputs are exactly the
five unique feature spaces represented by the nine saved NPE models.


In [ ]:
from lg_npe.data import load_flamingo_pair_table, load_tng_pair_table
from lg_npe.features import build_design_matrix
from lg_npe.splitting import pair_groups

source_table = load_flamingo_pair_table(PROJECT)
transfer_table = load_tng_pair_table(PROJECT)

source_designs = {}
transfer_designs = {}
feature_registry = {}
population_rows = []

for feature_space, mass_feature in FEATURE_SPACES.items():
    source_frame, source_names, source_target, source_counts = build_design_matrix(
        source_table.frame,
        target="m200c",
        feature_block="dynamics",
        mass_feature=mass_feature,
        validity_policy="global_complete_case",
    )
    transfer_frame, transfer_names, transfer_target, transfer_counts = build_design_matrix(
        transfer_table.frame,
        target="m200c",
        feature_block="dynamics",
        mass_feature=mass_feature,
        validity_policy="global_complete_case",
    )
    if source_names != transfer_names or source_target != transfer_target:
        raise RuntimeError(f"Source and transfer definitions differ for {feature_space}.")
    source_designs[feature_space] = source_frame
    transfer_designs[feature_space] = transfer_frame
    feature_registry[feature_space] = source_names
    population_rows.extend([
        {
            "feature_space": feature_space,
            "simulation": "flamingo",
            "n": len(source_frame),
            "same_host": int(source_frame["same_host"].sum()),
            "different_host": int((~source_frame["same_host"]).sum()),
            "feature_names": json.dumps(source_names),
            **{f"stage_{key}": value for key, value in source_counts.items()},
        },
        {
            "feature_space": feature_space,
            "simulation": "tng",
            "n": len(transfer_frame),
            "same_host": int(transfer_frame["same_host"].sum()),
            "different_host": int((~transfer_frame["same_host"]).sum()),
            "feature_names": json.dumps(transfer_names),
            **{f"stage_{key}": value for key, value in transfer_counts.items()},
        },
    ])

reference_source_pairs = source_designs["dynamics"]["pair_index"].to_numpy()
reference_transfer_pairs = transfer_designs["dynamics"]["pair_index"].to_numpy()
for feature_space in FEATURE_SPACES:
    if not np.array_equal(
        source_designs[feature_space]["pair_index"].to_numpy(), reference_source_pairs
    ):
        raise RuntimeError("FLAMINGO populations differ between feature spaces.")
    if not np.array_equal(
        transfer_designs[feature_space]["pair_index"].to_numpy(), reference_transfer_pairs
    ):
        raise RuntimeError("TNG populations differ between feature spaces.")

population_table = pd.DataFrame(population_rows)
population_table.to_csv(OUTPUT / "domain_population_registry.csv", index=False)
population_table[[
    "feature_space", "simulation", "n", "same_host", "different_host", "feature_names"
]]


## 3. Group-aware cross-fitted domain scores

The classifier is a modest histogram gradient-boosted tree. Its hyperparameters are
fixed before inspecting the results. Class weights give FLAMINGO and TNG equal total
weight despite their very different catalogue sizes.

Within each outer training fold, a separate group-aware validation fold is used for
Platt calibration. The calibration regression is also balanced, so the resulting
score is an equal-prior domain score. It is not the probability of drawing TNG from
the natural mixture of these two catalogues. This convention places the balanced
decision boundary at 0.5 and makes the proposed fuzzy region operational without
pretending that the catalogue prevalence is physically meaningful.


In [ ]:
def probability_logit(probability: np.ndarray) -> np.ndarray:
    clipped = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped)).reshape(-1, 1)


def check_disjoint(groups: np.ndarray, *indices: np.ndarray) -> None:
    sets = [set(groups[index]) for index in indices]
    for left in range(len(sets)):
        for right in range(left + 1, len(sets)):
            if sets[left] & sets[right]:
                raise RuntimeError("Connected host components overlap between splits.")


def cross_fitted_domain_scores(
    source_frame: pd.DataFrame,
    transfer_frame: pd.DataFrame,
    names: list[str],
    feature_space: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    source_X = source_frame[names].to_numpy(np.float64)
    transfer_X = transfer_frame[names].to_numpy(np.float64)
    X = np.vstack([source_X, transfer_X])
    y = np.concatenate([
        np.zeros(len(source_frame), dtype=np.int8),
        np.ones(len(transfer_frame), dtype=np.int8),
    ])
    if not np.isfinite(X).all():
        raise RuntimeError(f"Non-finite classifier input for {feature_space}.")

    source_groups = pair_groups(source_frame)
    transfer_groups = pair_groups(transfer_frame)
    groups = np.concatenate([
        np.array([f"flamingo:{value}" for value in source_groups], dtype=object),
        np.array([f"tng:{value}" for value in transfer_groups], dtype=object),
    ])

    scores = np.full(len(y), np.nan, dtype=np.float64)
    fold_assignment = np.full(len(y), -1, dtype=np.int16)
    fold_rows = []
    outer = StratifiedGroupKFold(
        n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED
    )

    for fold, (outer_train, outer_test) in enumerate(outer.split(X, y, groups)):
        inner = StratifiedGroupKFold(
            n_splits=VALIDATION_FOLDS,
            shuffle=True,
            random_state=RANDOM_SEED + 100 + fold,
        )
        fit_relative, validation_relative = next(
            inner.split(X[outer_train], y[outer_train], groups[outer_train])
        )
        fit_index = outer_train[fit_relative]
        validation_index = outer_train[validation_relative]
        check_disjoint(groups, fit_index, validation_index, outer_test)

        classifier = HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_iter=CLASSIFIER_MAX_ITER,
            max_leaf_nodes=15,
            min_samples_leaf=30,
            l2_regularization=1.0,
            early_stopping=False,
            class_weight="balanced",
            random_state=RANDOM_SEED + fold,
        )
        classifier.fit(X[fit_index], y[fit_index])

        validation_raw = classifier.predict_proba(X[validation_index])[:, 1]
        calibrator = LogisticRegression(
            C=1e6,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=500,
            random_state=RANDOM_SEED + 500 + fold,
        )
        calibrator.fit(probability_logit(validation_raw), y[validation_index])

        test_raw = classifier.predict_proba(X[outer_test])[:, 1]
        test_score = calibrator.predict_proba(probability_logit(test_raw))[:, 1]
        scores[outer_test] = test_score
        fold_assignment[outer_test] = fold

        fold_rows.append({
            "feature_space": feature_space,
            "fold": fold,
            "n_fit": len(fit_index),
            "n_validation": len(validation_index),
            "n_test": len(outer_test),
            "fit_tng": int(y[fit_index].sum()),
            "validation_tng": int(y[validation_index].sum()),
            "test_tng": int(y[outer_test].sum()),
            "validation_auc_raw": roc_auc_score(
                y[validation_index], validation_raw
            ),
            "test_auc": roc_auc_score(y[outer_test], test_score),
            "test_balanced_accuracy": balanced_accuracy_score(
                y[outer_test], test_score >= 0.5
            ),
        })

    if not np.isfinite(scores).all() or (fold_assignment < 0).any():
        raise RuntimeError(f"Incomplete out-of-fold predictions for {feature_space}.")

    score_table = pd.DataFrame({
        "feature_space": feature_space,
        "simulation": np.where(y == 1, "tng", "flamingo"),
        "domain_label": y,
        "pair_index": np.concatenate([
            source_frame["pair_index"].to_numpy(),
            transfer_frame["pair_index"].to_numpy(),
        ]),
        "same_host": np.concatenate([
            source_frame["same_host"].to_numpy(bool),
            transfer_frame["same_host"].to_numpy(bool),
        ]),
        "connected_component": groups,
        "outer_fold": fold_assignment,
        "domain_score": scores,
    })
    return score_table, pd.DataFrame(fold_rows)


In [ ]:
score_parts = []
fold_parts = []

for feature_number, feature_space in enumerate(FEATURE_SPACES, start=1):
    print(f"[{feature_number}/{len(FEATURE_SPACES)}] {feature_space}")
    scores, folds = cross_fitted_domain_scores(
        source_designs[feature_space],
        transfer_designs[feature_space],
        feature_registry[feature_space],
        feature_space,
    )
    score_parts.append(scores)
    fold_parts.append(folds)
    del scores, folds
    gc.collect()

domain_scores = pd.concat(score_parts, ignore_index=True)
fold_metrics = pd.concat(fold_parts, ignore_index=True)
domain_scores.to_csv(OUTPUT / "domain_scores_oof.csv.gz", index=False)
fold_metrics.to_csv(OUTPUT / "domain_classifier_fold_metrics.csv", index=False)

classifier_rows = []
for feature_space, frame in domain_scores.groupby("feature_space", sort=False):
    y = frame["domain_label"].to_numpy()
    score = frame["domain_score"].to_numpy()
    classifier_rows.append({
        "feature_space": feature_space,
        "n_flamingo": int((y == 0).sum()),
        "n_tng": int((y == 1).sum()),
        "roc_auc": roc_auc_score(y, score),
        "balanced_accuracy_at_0.5": balanced_accuracy_score(y, score >= 0.5),
        "median_score_flamingo": float(np.median(score[y == 0])),
        "median_score_tng": float(np.median(score[y == 1])),
    })
classifier_summary = pd.DataFrame(classifier_rows)
classifier_summary.to_csv(OUTPUT / "domain_classifier_summary.csv", index=False)
classifier_summary.round(4)


In [ ]:
score_bins = np.linspace(0, 1, 41)
fig, axes = plt.subplots(len(FEATURE_SPACES), 1, figsize=(9, 13), sharex=True)

for ax, feature_space in zip(axes, FEATURE_SPACES):
    current = domain_scores.loc[domain_scores["feature_space"] == feature_space]
    for simulation, color in (("flamingo", "tab:blue"), ("tng", "tab:orange")):
        values = current.loc[
            current["simulation"] == simulation, "domain_score"
        ].to_numpy()
        ax.hist(
            values,
            bins=score_bins,
            density=True,
            histtype="step",
            linewidth=1.8,
            color=color,
            label=simulation.upper(),
        )
    ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
    ax.axvspan(0.4, 0.6, color="grey", alpha=0.12)
    auc = classifier_summary.loc[
        classifier_summary["feature_space"] == feature_space, "roc_auc"
    ].iloc[0]
    ax.set_ylabel("Density")
    ax.set_title(f"{feature_space.replace('_', ' ').title()}   AUC={auc:.3f}")
    ax.grid(axis="y", alpha=0.2)

axes[0].legend(loc="upper center", ncol=2)
axes[-1].set_xlabel("Balanced-prior out-of-fold domain score")
fig.suptitle("Held-out FLAMINGO–TNG domain scores", y=0.995)
fig.tight_layout()
fig.savefig(OUTPUT / "domain_score_distributions.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "domain_score_distributions.pdf", bbox_inches="tight")
plt.show()


## 4. Retained common-support populations

Retention is evaluated on the same populations used for posterior assessment: the
21,409 held-out FLAMINGO pairs saved by the follow-up run and all 1,530 TNG pairs.
Counts are reported before posterior metrics so a seemingly improved result cannot be
separated from the number and type of systems retained.


In [ ]:
reference_source_summary = pd.read_csv(
    POSTERIOR_EVALUATION
    / "m200c_bound_ratio"
    / "source_heldout_posterior_summary.csv.gz",
    usecols=["pair_index"],
)
source_evaluation_pairs = set(reference_source_summary["pair_index"].to_numpy())

classifier_evaluation = domain_scores.loc[
    (domain_scores["simulation"] == "tng")
    | (
        (domain_scores["simulation"] == "flamingo")
        & domain_scores["pair_index"].isin(source_evaluation_pairs)
    )
].copy()

subset_masks = {
    "pooled": lambda frame: np.ones(len(frame), dtype=bool),
    "same_host": lambda frame: frame["same_host"].to_numpy(bool),
    "different_host": lambda frame: ~frame["same_host"].to_numpy(bool),
}


def overlap_label(half_width: float) -> str:
    if half_width >= 0.5:
        return "all"
    return f"{0.5 - half_width:.2f}-{0.5 + half_width:.2f}"


retention_rows = []
for feature_space, feature_frame in classifier_evaluation.groupby(
    "feature_space", sort=False
):
    for simulation, simulation_frame in feature_frame.groupby("simulation", sort=False):
        for subset, mask_function in subset_masks.items():
            base = simulation_frame.loc[mask_function(simulation_frame)]
            for half_width in OVERLAP_HALF_WIDTHS:
                keep = np.abs(base["domain_score"].to_numpy() - 0.5) <= half_width
                retention_rows.append({
                    "feature_space": feature_space,
                    "simulation": simulation,
                    "subset": subset,
                    "half_width": half_width,
                    "score_interval": overlap_label(half_width),
                    "n_total": len(base),
                    "n_retained": int(keep.sum()),
                    "retained_fraction": float(keep.mean()) if len(base) else np.nan,
                })

retention_table = pd.DataFrame(retention_rows)
retention_table.to_csv(OUTPUT / "overlap_retention.csv", index=False)
retention_table.loc[
    (retention_table["half_width"].isin([0.10, 0.20, 0.50]))
    & (retention_table["simulation"] == "tng")
].reset_index(drop=True)


In [ ]:
fig, axes = plt.subplots(1, len(FEATURE_SPACES), figsize=(18, 4), sharey=True)
style = {
    ("flamingo", "pooled"): ("tab:blue", "-"),
    ("tng", "pooled"): ("tab:orange", "-"),
    ("tng", "same_host"): ("tab:green", "--"),
    ("tng", "different_host"): ("tab:red", ":"),
}

for ax, feature_space in zip(axes, FEATURE_SPACES):
    current = retention_table.loc[retention_table["feature_space"] == feature_space]
    for (simulation, subset), (color, linestyle) in style.items():
        line = current.loc[
            (current["simulation"] == simulation) & (current["subset"] == subset)
        ].sort_values("half_width")
        ax.plot(
            line["half_width"],
            line["retained_fraction"],
            marker="o",
            color=color,
            linestyle=linestyle,
            label=f"{simulation} {subset}",
        )
    ax.set_title(feature_space.replace("_", " ").title())
    ax.set_xlabel("Overlap half-width about 0.5")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Retained fraction")
axes[-1].legend(loc="lower right", fontsize=8)
fig.suptitle("Population retained by configurable domain-score intervals", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT / "overlap_retention.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT / "overlap_retention.pdf", bbox_inches="tight")
plt.show()


## 5. Join domain scores to the saved posteriors

The join uses the stable pair index and is validated one-to-one. Posterior metrics are
recomputed from the exported median, intervals, rank and standard deviation; no saved
model is loaded. In addition to predictive and calibration metrics, the table records
the target mean, standard deviation and range. This matters because restricting
domain score can change the target population and make an RMSE change descriptive rather
than a like-for-like causal comparison.


In [ ]:
joined_parts = []
for model_label, feature_space in MODEL_TO_FEATURE_SPACE.items():
    for population, simulation in (
        ("source_heldout", "flamingo"),
        ("transfer_tng", "tng"),
    ):
        posterior_path = (
            POSTERIOR_EVALUATION
            / model_label
            / f"{population}_posterior_summary.csv.gz"
        )
        posterior = pd.read_csv(posterior_path)
        domain = domain_scores.loc[
            (domain_scores["feature_space"] == feature_space)
            & (domain_scores["simulation"] == simulation),
            ["pair_index", "domain_score", "outer_fold"],
        ]
        merged = posterior.merge(
            domain, on="pair_index", how="left", validate="one_to_one"
        )
        if len(merged) != len(posterior) or merged["domain_score"].isna().any():
            raise RuntimeError(
                f"Domain-score join failed for {model_label}/{population}."
            )
        merged.insert(0, "model", model_label)
        merged.insert(1, "target", MODEL_TARGET[model_label])
        merged.insert(2, "feature_space", feature_space)
        merged.insert(3, "population", population)
        joined_parts.append(merged)

posterior_domain = pd.concat(joined_parts, ignore_index=True)
posterior_domain.to_csv(OUTPUT / "posterior_domain_join.csv.gz", index=False)
print(f"Joined {len(posterior_domain):,} model-pair rows.")


In [ ]:
def metrics_from_summary(frame: pd.DataFrame) -> dict:
    if len(frame) == 0:
        return {
            key: np.nan
            for key in (
                "rmse", "bias", "abs_bias", "r2", "coverage68", "coverage95",
                "median_beta", "rank_cdf_distance", "mean_width68", "mean_width95",
                "truth_mean", "truth_std", "truth_min", "truth_max",
                "mean_domain_score",
            )
        }

    truth = frame["truth"].to_numpy()
    residual = frame["posterior_median"].to_numpy() - truth
    n_draw_values = frame["n_draws"].unique()
    if len(n_draw_values) != 1:
        raise RuntimeError("A selected table contains more than one draw count.")
    n_draws = int(n_draw_values[0])
    ranks = frame["rank"].to_numpy(np.int64)
    rank_counts = np.bincount(ranks, minlength=n_draws + 1)
    rank_reference_cdf = np.arange(1, n_draws + 2) / (n_draws + 1)
    bias = float(np.mean(residual))
    return {
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "bias": bias,
        "abs_bias": abs(bias),
        "r2": float(r2_score(truth, frame["posterior_median"]))
        if len(frame) > 1 else np.nan,
        "coverage68": float(
            np.mean((truth >= frame["q16"]) & (truth <= frame["q84"]))
        ),
        "coverage95": float(
            np.mean((truth >= frame["q025"]) & (truth <= frame["q975"]))
        ),
        "median_beta": float(np.median(frame["beta"])),
        "rank_cdf_distance": float(
            np.max(
                np.abs(
                    rank_counts.cumsum() / len(frame) - rank_reference_cdf
                )
            )
        ),
        "mean_width68": float(np.mean(frame["q84"] - frame["q16"])),
        "mean_width95": float(np.mean(frame["q975"] - frame["q025"])),
        "truth_mean": float(np.mean(truth)),
        "truth_std": float(np.std(truth)),
        "truth_min": float(np.min(truth)),
        "truth_max": float(np.max(truth)),
        "mean_domain_score": float(np.mean(frame["domain_score"])),
    }


metric_rows = []
group_columns = ["model", "target", "feature_space", "population"]
for keys, model_frame in posterior_domain.groupby(group_columns, sort=False):
    model_label, target, feature_space, population = keys
    for subset, mask_function in subset_masks.items():
        base = model_frame.loc[mask_function(model_frame)]
        for half_width in OVERLAP_HALF_WIDTHS:
            selected = base.loc[
                np.abs(base["domain_score"].to_numpy() - 0.5) <= half_width
            ]
            metric_rows.append({
                "model": model_label,
                "target": target,
                "feature_space": feature_space,
                "population": population,
                "subset": subset,
                "half_width": half_width,
                "score_interval": overlap_label(half_width),
                "n_total": len(base),
                "n": len(selected),
                "retained_fraction": len(selected) / len(base) if len(base) else np.nan,
                **metrics_from_summary(selected),
            })

overlap_metrics = pd.DataFrame(metric_rows)
overlap_metrics.to_csv(OUTPUT / "overlap_posterior_metrics.csv", index=False)

overlap_metrics.loc[
    (overlap_metrics["population"] == "transfer_tng")
    & (overlap_metrics["subset"].isin(["pooled", "different_host"]))
    & (overlap_metrics["half_width"].isin([0.10, 0.20, 0.50])),
    [
        "model", "target", "subset", "score_interval", "n", "retained_fraction",
        "rmse", "bias", "coverage68", "coverage95", "truth_std",
    ],
].round(4).reset_index(drop=True)


In [ ]:
DISPLAY_NAMES = {
    "m200c_bound_ratio": "bound ratio",
    "m200c_dynamics": "dynamics",
    "m200c_stellar_individual": "stellar individual",
    "m200c_stellar_ratio": "stellar ratio",
    "m200c_stellar_sum": "stellar sum",
    "bound_sum_dynamics": "dynamics",
    "bound_sum_stellar_individual": "stellar individual",
    "bound_sum_stellar_ratio": "stellar ratio",
    "bound_sum_stellar_sum": "stellar sum",
}


def plot_target_sensitivity(target: str) -> None:
    current = overlap_metrics.loc[
        (overlap_metrics["population"] == "transfer_tng")
        & (overlap_metrics["target"] == target)
    ]
    model_order = list(dict.fromkeys(current["model"]))
    subsets = ["pooled", "same_host", "different_host"]
    metrics_to_plot = [
        ("rmse", "RMSE [dex]", None),
        ("bias", "Bias [dex]", 0.0),
        ("coverage68", "68% coverage", 0.68),
    ]
    fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharex=True)
    for row, subset in enumerate(subsets):
        for column, (metric, ylabel, reference) in enumerate(metrics_to_plot):
            ax = axes[row, column]
            for model_label in model_order:
                line = current.loc[
                    (current["subset"] == subset)
                    & (current["model"] == model_label)
                ].sort_values("half_width")
                ax.plot(
                    line["half_width"],
                    line[metric],
                    marker="o",
                    label=DISPLAY_NAMES[model_label],
                )
            if reference is not None:
                ax.axhline(reference, color="black", linestyle="--", linewidth=1)
            ax.grid(alpha=0.25)
            if row == 0:
                ax.set_title(ylabel)
            if column == 0:
                ax.set_ylabel(subset.replace("_", " ").title())
            if row == 2:
                ax.set_xlabel("Overlap half-width about 0.5")
    axes[0, -1].legend(loc="best", fontsize=8)
    title = "M200c" if target == "m200c" else "Combined bound-subhalo mass"
    fig.suptitle(f"TNG transfer metrics inside configurable support: {title}", y=0.995)
    fig.tight_layout()
    stem = f"tng_overlap_metrics_{target}"
    fig.savefig(OUTPUT / f"{stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(OUTPUT / f"{stem}.pdf", bbox_inches="tight")
    plt.show()


plot_target_sensitivity("m200c")
plot_target_sensitivity("bound_subhalo_sum")


## 6. Interpretation rules

The classifier AUC answers whether the simulations are separable in a chosen feature
space. It does not measure NPE accuracy. The overlap-conditioned posterior tables then
ask whether predictive bias and calibration change among systems that the classifier
finds difficult to assign.

Three checks should be made together:

1. Inspect retained counts and host composition. Do not interpret a narrow interval
 with only a small number of different-host TNG pairs.
2. Inspect the target standard deviation. A lower RMSE can follow from a
 narrower target population without representing better transport.
3. Compare bias, coverage and posterior width alongside RMSE. Improvement in one metric
 is not sufficient to call a representation portable.

Persistence of bias at narrow score intervals is evidence against lack of marginal
support as a complete explanation. It is not, by itself, proof of conditional concept
shift because the classifier is finite, feature-set dependent and imperfectly
calibrated. Conversely, reduced bias inside overlap does not establish that
extrapolation is the only failure mode.

The next decision should be made only after these sensitivity curves are inspected.
An overlap-only NPE retrain remains a separate experiment and should not be introduced
into this first diagnostic pass.


## 7. Output manifest

The manifest records the classifier design, feature names, package versions and every
output created by this notebook. It also records the fixed posterior export used for
the join so that later analysis does not silently switch between result generations.


In [ ]:
output_manifest = {
    "experiment_id": f"domain_common_support_{timestamp}",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project": str(PROJECT),
    "git_revision": git_revision,
    "output": str(OUTPUT),
    "purpose": (
        "Held-out FLAMINGO-TNG domain classification and common-support "
        "evaluation; no NPE training"
    ),
    "notebook": str(
        PROJECT / "notebooks" / "diagnostics" / "domain_common_support.ipynb"
    ),
    "posterior_evaluation": str(POSTERIOR_EVALUATION),
    "posterior_evaluation_manifest_sha256": posterior_manifest_sha256,
    "versions": versions,
    "settings": {
        "random_seed": RANDOM_SEED,
        "outer_folds": OUTER_FOLDS,
        "validation_folds": VALIDATION_FOLDS,
        "classifier": "HistGradientBoostingClassifier",
        "classifier_hyperparameters": {
            "learning_rate": 0.06,
            "max_iter": CLASSIFIER_MAX_ITER,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 30,
            "l2_regularization": 1.0,
            "early_stopping": False,
        },
        "class_weight": "balanced",
        "calibration": "balanced Platt calibration on an inner group-aware split",
        "overlap_half_widths": OVERLAP_HALF_WIDTHS,
        "validity_policy": "global_complete_case",
    },
    "feature_spaces": feature_registry,
    "model_to_feature_space": MODEL_TO_FEATURE_SPACE,
    "reporting_notes": [
        "All domain scores are outer-fold held out by connected host component.",
        "The score is calibrated under equal simulation weights and is not a natural catalogue-prevalence probability.",
        "No overlap interval is treated as scientifically authoritative.",
        "Different-host R2 is retained in machine-readable output but is not used for ranking.",
        "Overlap-conditioned metrics are descriptive and can change with target-population composition.",
        "No NPE was fitted or updated in this notebook.",
    ],
    "outputs": sorted(
        str(path.relative_to(OUTPUT))
        for path in OUTPUT.rglob("*")
        if path.is_file()
    ),
}
(OUTPUT / "domain_overlap_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

print(f"Completed domain-overlap export: {OUTPUT}")
print("No NPE was trained or updated.")
